## schem_loader_py_test


In [ ]:
import sys

In [ ]:
print(sys.executable)

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))


In [ ]:
from pathlib import Path
from replay.service.schema_loader import MasterSchema

schema = MasterSchema(
    Path(r"C:\streaming_emulator\contracts\master.json")
)

print("Contract version:", schema.contract_version)
print("Reference SIM:", schema.reference_sim)
print("Modules:", list(schema.modules.keys()))
print("Metadata fields:", list(schema.metadata_schema.keys()))
print("Validation rules:", schema.validation_rules)


## source_discovery_py_test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.schema_loader import MasterSchema
from replay.service.source_discovery import discover_sources

schema = MasterSchema(
    PIPELINE_ROOT / "contracts" / "master.json"
)

sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001", "sim003", "sim007"],
)

print(f"Discovered {len(sources)} sources:\n")
for s in sources:
    print(
        f"vehicle={s['vehicle_id']:<6} "
        f"module={s['module']:<13} "
        f"source_id={s['source_id']:<20} "
        f"path={s['csv_path'].name}"
    )


## dlq_py_test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.dlq import DLQWriter

dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")

fake_row = {
    "timestamp": "2024-07-05T08:00:00Z",
    "engine_rpm": "NOT_A_NUMBER"
}

dlq.write(
    source_id="sim999_engine",
    vehicle_id="sim999",
    module="engine",
    row_index=42,
    error_type="schema_validation",
    error_message="engine_rpm is not a valid float",
    error_details={"expected_type": "float64"},
    raw_row=fake_row,
)

print("DLQ test record written.")


## worker_py_test


#### row hashing test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.worker import compute_row_hash

features_a = {
    "engine_rpm": 1200.0,
    "engine_temp": 85.2,
}

features_b = {
    "engine_temp": 85.2,
    "engine_rpm": 1200.0,  
}

h1 = compute_row_hash(
    vehicle_id="sim001",
    module="engine",
    timestamp="2024-07-05 08:00:00+00:00",
    features=features_a,
)

h2 = compute_row_hash(
    vehicle_id="sim001",
    module="engine",
    timestamp="2024-07-05 08:00:00+00:00",
    features=features_b,
)

print("Hash A:", h1)
print("Hash B:", h2)
print("Hashes equal:", h1 == h2)


In [ ]:
h3 = compute_row_hash(
    vehicle_id="sim001",
    module="engine",
    timestamp="2024-07-05 08:00:01+00:00",  # different timestamp
    features=features_a,
)

print("Different timestamp hash differs:", h1 != h3)


#### resume logic test

In [ ]:
# fresh read test
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.worker import read_csv_rows

csv_path = (
    PIPELINE_ROOT
    / "data"
    / "vehicles"
    / "sim001"
    / "synthetic_engine_inference_scenarioA_sim001.csv"
)

rows = read_csv_rows(csv_path=csv_path, start_row_index=-1)

for i, (idx, row) in enumerate(rows):
    print(idx, list(row.keys())[:3])
    if i == 2:
        break


In [ ]:
# resume logic test
rows = read_csv_rows(csv_path=csv_path, start_row_index=5)

for i, (idx, row) in enumerate(rows):
    print("Resumed at index:", idx)
    break


In [ ]:
# kill and resume simulation test
rows = read_csv_rows(csv_path=csv_path, start_row_index=-1)
processed = []

for idx, row in rows:
    processed.append(idx)
    if len(processed) == 3:
        break

last_idx = processed[-1]
print("Last processed:", last_idx)

# Resume
rows = read_csv_rows(csv_path=csv_path, start_row_index=last_idx)
print("Next after resume:", next(rows)[0])


#### combined worker loop test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))


In [ ]:
from replay.service.schema_loader import MasterSchema
from replay.service.validator import SchemaValidator
from replay.service.checkpoint import CheckpointStore
from replay.service.dlq import DLQWriter
from replay.service.source_discovery import discover_sources
from replay.service.worker import run_worker_once

schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)
checkpoints = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")
dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")

sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001"],
)

metrics = run_worker_once(
    source=sources[0],
    schema_validator=validator,
    checkpoint_store=checkpoints,
    dlq_writer=dlq,
)

print(metrics)


In [ ]:
# rerun the same code : expected rows_seen = 0
from replay.service.schema_loader import MasterSchema
from replay.service.validator import SchemaValidator
from replay.service.checkpoint import CheckpointStore
from replay.service.dlq import DLQWriter
from replay.service.source_discovery import discover_sources
from replay.service.worker import run_worker_once

schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)
checkpoints = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")
dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")

sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001"],
)

metrics = run_worker_once(
    source=sources[0],
    schema_validator=validator,
    checkpoint_store=checkpoints,
    dlq_writer=dlq,
)

print(metrics)


#### fixed_rate and batch mode test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.schema_loader import MasterSchema
from replay.service.source_discovery import discover_sources
from replay.service.validator import SchemaValidator
from replay.service.checkpoint import CheckpointStore
from replay.service.dlq import DLQWriter
from replay.service.worker import run_worker_with_replay_mode


schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)
checkpoints = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")
dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")


sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001"],  
)

print(f"Discovered {len(sources)} sources")
print(sources[0])


In [ ]:
metrics = run_worker_with_replay_mode(
    source=sources[0],
    schema_validator=validator,
    checkpoint_store=checkpoints,
    dlq_writer=dlq,
    mode="fixed_rate",
    rows_per_second=5,
)

print(metrics)

In [ ]:
metrics = run_worker_with_replay_mode(
    source=sources[0],
    schema_validator=validator,
    checkpoint_store=checkpoints,
    dlq_writer=dlq,
    mode="batch",
    flush_interval_seconds=3,
)

print(metrics)


## http_client_py_test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.schema_loader import MasterSchema
from replay.service.source_discovery import discover_sources
from replay.service.validator import SchemaValidator
from replay.service.checkpoint import CheckpointStore
from replay.service.dlq import DLQWriter
from replay.service.worker import run_worker_with_replay_mode


schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)
checkpoints = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")
dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")


sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001"],  
)

print(f"Discovered {len(sources)} sources")
print(sources[0])


In [ ]:
from replay.service.http_client import HttpClient

client = HttpClient(
    endpoint="http://127.0.0.1:8000/ingest/test",
    timeout_seconds=2,
    max_retries=3,
)

payload = {"hello": "world"}

resp = client.post_json(payload)
print(resp.status_code)

In [ ]:
client = HttpClient(
    endpoint="http://127.0.0.1:9999/nowhere",
    timeout_seconds=1,
    max_retries=3,
)

try:
    client.post_json({"test": 1})
except Exception as e:
    print(type(e), e)


## metrics_py_test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.schema_loader import MasterSchema
from replay.service.source_discovery import discover_sources
from replay.service.validator import SchemaValidator
from replay.service.checkpoint import CheckpointStore
from replay.service.dlq import DLQWriter
from replay.service.worker import run_worker_with_replay_mode


schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)
checkpoints = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")
dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")


sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001"],  
)

print(f"Discovered {len(sources)} sources")
print(sources[0])


In [ ]:
from replay.service.metrics import (
    rows_attempted_total,
    rows_sent_total,
    active_sources,
    batch_latency_ms,
    export_metrics,
)

rows_attempted_total.inc()
rows_sent_total.inc(2)

active_sources.inc()
active_sources.dec()

with batch_latency_ms.time():
    import time
    time.sleep(0.05)

print(export_metrics().decode())


## replay_service_py_test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.schema_loader import MasterSchema
from replay.service.source_discovery import discover_sources
from replay.service.validator import SchemaValidator
from replay.service.checkpoint import CheckpointStore
from replay.service.dlq import DLQWriter
from replay.service.worker import run_worker_with_replay_mode


schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)
checkpoints = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")
dlq = DLQWriter(PIPELINE_ROOT / "replay" / "dlq")


sources = discover_sources(
    pipeline_root=PIPELINE_ROOT,
    schema=schema,
    enabled_sims=["sim001"],  
)

print(f"Discovered {len(sources)} sources")
print(sources[0])


In [ ]:
from pathlib import Path
from replay.service.replay_service import ReplayService

service = ReplayService(
    pipeline_root=Path(r"C:\streaming_emulator"),
    enabled_sims=["sim001", "sim002"],
    replay_mode="fixed_rate",
    rows_per_second=10,
)

service.start()
service.wait()   


In [ ]:
from replay.service.metrics import active_sources

print(active_sources._value.get())


In [ ]:
from pathlib import Path
from replay.service.schema_loader import MasterSchema

schema = MasterSchema(
    Path(r"C:\streaming_emulator\contracts\master.json")
)

print(schema.contract_version)
print(schema.reference_sim)


In [ ]:
from replay.service.source_discovery import discover_sources

sources = discover_sources(
    pipeline_root=Path(r"C:\streaming_emulator"),
    schema=schema,
    enabled_sims=["sim001", "sim002"],
)

print(len(sources))
for s in sources[:3]:
    print(s)


In [ ]:
from pathlib import Path
from replay.service.replay_service import ReplayService
service = ReplayService(
    pipeline_root=Path(r"C:\streaming_emulator"),
    enabled_sims=["sim001", "sim002"],
    replay_mode="fixed_rate",
    rows_per_second=10,
)

service.start(reset=True)
service.wait()


In [ ]:
from pathlib import Path
from replay.service.replay_service import ReplayService
service = ReplayService(
    pipeline_root=Path(r"C:\streaming_emulator"),
    enabled_sims=["sim001", "sim002"],
    replay_mode="fixed_rate",
    rows_per_second=10,
)

service.start(reset=False)
service.wait()

## validator_py_test

In [ ]:
from pathlib import Path
import pandas as pd
from collections import OrderedDict

ENGINE_CSV = Path(
    r"C:\streaming_emulator\data\vehicles\sim001"
) / "synthetic_engine_inference_scenarioA_sim001.csv"

if not ENGINE_CSV.exists():
    raise FileNotFoundError(f"Engine CSV not found: {ENGINE_CSV}")


df = pd.read_csv(ENGINE_CSV, low_memory=False)

if df.empty:
    raise RuntimeError("Engine CSV is empty; cannot create valid_row.")

row = df.dropna().iloc[0]


valid_row = OrderedDict()

for col in df.columns:
    value = row[col]

    
    if pd.isna(value):
        valid_row[col] = None
    elif hasattr(value, "item"):
        valid_row[col] = value.item()
    else:
        valid_row[col] = value


if "source_id" in valid_row:
    valid_row["source_id"] = "sim001"



valid_row


In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.schema_loader import MasterSchema
from replay.service.validator import SchemaValidator, RowValidationError


In [ ]:
schema = MasterSchema(PIPELINE_ROOT / "contracts" / "master.json")
validator = SchemaValidator(schema)

validated = validator.validate_row(
    module="engine",
    row=valid_row,
)

print("Validated OK:", type(validated))


In [ ]:
bad_row = valid_row.copy()
bad_row.pop("engine_oil_temperature")

try:
    validator.validate_row(module="engine", row=bad_row)
except RowValidationError as e:
    print("Error:", e.message)
    print("Details:", e.details)


In [ ]:
bad_row = valid_row.copy()
bad_row["unexpected"] = 123

try:
    validator.validate_row(module="engine", row=bad_row)
except RowValidationError as e:
    print("Extra field rejected")


In [ ]:
bad_row = dict(reversed(list(valid_row.items())))

try:
    validator.validate_row(module="engine", row=bad_row)
except RowValidationError as e:
    print("Order enforced")


## checkpoint_py test

In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from replay.service.checkpoint import CheckpointStore

store = CheckpointStore(PIPELINE_ROOT / "replay" / "checkpoints")

SOURCE_ID = "sim_test_engine"

print("Initial load:", store.load(SOURCE_ID))

store.save(
    source_id=SOURCE_ID,
    last_row_index=100,
    last_row_hash="hash_100",
)


cp = store.load(SOURCE_ID)
print("Reloaded:", cp)


store.save(
    source_id=SOURCE_ID,
    last_row_index=200,
    last_row_hash="hash_200",
)

print("Updated:", store.load(SOURCE_ID))


store.reset(SOURCE_ID)

print("After reset:", store.load(SOURCE_ID))
